# 🧬 Thực Nghiệm 06: Kiến Trúc Lai (Hybrid Architecture) & Platt Scaling
**Quy tắc**: `BR-02: Hybrid Scoring` & `BR-05: Platt Brier Score <= 0.05`.


In [ ]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from xgboost import XGBClassifier
from sklearn.ensemble import IsolationForest
from sklearn.calibration import CalibratedClassifierCV, calibration_curve
from sklearn.metrics import brier_score_loss, precision_recall_curve, auc
from scripts.train_xgboost_real import generate_training_data
X, y = generate_training_data(n_normal=20000, n_fraud=1500)
xgb = XGBClassifier(n_estimators=100, max_depth=5, scale_pos_weight=13, eval_metric='aucpr', random_state=42).fit(X, y)
iso = IsolationForest(n_estimators=100, contamination=0.05, random_state=42).fit(X)
p_xgb = xgb.predict_proba(X)[:, 1]
raw_anom = -iso.score_samples(X); p_iso = (raw_anom - raw_anom.min()) / (raw_anom.max() - raw_anom.min())
p_hyb = 0.7 * p_xgb + 0.3 * p_iso
p1, r1, _ = precision_recall_curve(y, p_xgb); p3, r3, _ = precision_recall_curve(y, p_hyb)
print(f'XGB PR-AUC: {auc(r1, p1):.4f} | Hybrid PR-AUC: {auc(r3, p3):.4f}')


### 2. Platt Scaling Calibration (BR-05)


In [ ]:
cal = CalibratedClassifierCV(xgb, method='sigmoid', cv='prefit').fit(X, y)
p_cal = cal.predict_proba(X)[:, 1]
print(f'Brier Score Before: {brier_score_loss(y, p_xgb):.4f} | After: {brier_score_loss(y, p_cal):.4f} (<= 0.05)')
